# Landmark-Pair Fingerprinting Demo

## MinHash vs Landmark-Pair Fingerprinting Benchmark

This notebook demonstrates landmark-pair fingerprinting for near-duplicate text detection, benchmarked against MinHash (Jaccard & Containment) and SimHash on GLUE MRPC paraphrase pairs + synthetic structural edits.

**Key idea:** Combine high-salience (TF-IDF) token pairs with quantized relative positions — inspired by Shazam's audio fingerprinting — to create structural signatures robust to insertions, deletions, and reordering.

## Setup: Install Dependencies

Install core packages at versions matching Colab's environment, plus non-Colab dependencies.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')

# Core packages: pre-installed on Colab, install locally at Colab versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'scikit-learn==1.6.1', 'matplotlib==3.10.0')

## Imports

In [ ]:
import gc
import hashlib
import json
import math
import random
import re
import time
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import precision_recall_curve, average_precision_score

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

random.seed(42)
np.random.seed(42)

## Data Loading

Load mini demo data from GitHub (with fallback to local).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-cb9424-landmark-pair-fingerprinting-for-text-cr/main/round-2/evaluation-1/demo/mini_demo_data.json"
import os

def load_data():
    """Load mini demo data from GitHub or local fallback."""
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL, timeout=5) as response:
            return json.loads(response.read().decode())
    except Exception:
        pass
    
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f:
            return json.load(f)
    
    raise FileNotFoundError("Could not load mini_demo_data.json from GitHub or local path")

data = load_data()
print(f"Loaded data: {data['metadata']['total_pairs']} total pairs ({data['metadata']['mrpc_pairs']} MRPC, {data['metadata']['synthetic_pairs']} synthetic)")

## Config: Tunable Parameters

Start with minimal parameters for fast demo. Increase to reproduce full results.

In [ ]:
# ─── Fingerprinting parameters ───
TOP_K = 20              # Top-K high-TF-IDF tokens as landmarks
WINDOW = 30             # Co-occurrence window size (positions)
QUANTIZE = 5            # Quantize position delta to this granularity

# ─── MinHash parameters ───
MINHASH_NUM_PERM = 64   # Number of hash functions (64 for demo, 128 for full)
SHINGLE_SIZE = 5        # Character shingle size

# ─── Demo data subsetting ───
# Use all demo examples (just 5)
N_DEMO_EXAMPLES = min(len(data['datasets'][0]['examples']), 5)

print(f"Config: TOP_K={TOP_K}, WINDOW={WINDOW}, QUANTIZE={QUANTIZE}")
print(f"Demo will process {N_DEMO_EXAMPLES} examples")

## Tokenization and IDF

Simple whitespace + regex-based tokenizer, compute TF-IDF for landmark selection.

In [ ]:
def tokenize(text: str) -> list:
    """Simple whitespace + punctuation tokenizer."""
    return re.findall(r"[a-z0-9]+", text.lower())

def compute_idf(corpus: list) -> dict:
    """Compute IDF scores: log((N+1)/(df+1)) + 1."""
    N = len(corpus)
    df = defaultdict(int)
    for tokens in corpus:
        for t in set(tokens):
            df[t] += 1
    return {t: math.log((N + 1) / (d + 1)) + 1 for t, d in df.items()}

# Prepare demo data: extract sentence pairs and tokenize
examples = data['datasets'][0]['examples'][:N_DEMO_EXAMPLES]
sentence_pairs = []
all_tokens = []

for ex in examples:
    inp = json.loads(ex['input'])
    s1, s2 = inp['sentence1'], inp['sentence2']
    sentence_pairs.append((s1, s2, int(ex['output'])))
    all_tokens.extend([tokenize(s1), tokenize(s2)])

idf = compute_idf(all_tokens)
print(f"IDF vocab size: {len(idf)}")
print(f"Example tokens from first pair: {all_tokens[0][:10]}")

## Landmark-Pair Fingerprinting

Extract high-TF-IDF token pairs and hash them with quantized position deltas.

In [ ]:
def extract_landmarks(tokens: list, idf: dict, top_k: int = 20) -> list:
    """Extract top-K high-TF-IDF tokens as landmarks (position, token, score)."""
    if not tokens:
        return []
    scores = [(i, t, idf.get(t, 0.0)) for i, t in enumerate(tokens)]
    scores.sort(key=lambda x: -x[2])
    selected = sorted(scores[:top_k], key=lambda x: x[0])  # Re-sort by position
    return selected

def fingerprint_landmark_pair(tokens: list, idf: dict,
                               top_k: int = 20, window: int = 30,
                               quantize: int = 5,
                               include_delta: bool = True) -> set:
    """Generate Shazam-inspired landmark-pair fingerprint."""
    landmarks = extract_landmarks(tokens, idf, top_k)
    if len(landmarks) < 2:
        return set()
    fp = set()
    for i, (pos_a, tok_a, _) in enumerate(landmarks):
        for pos_t, tok_t, _ in landmarks[i+1:]:
            if pos_t > pos_a + window:
                break
            if include_delta:
                delta = ((pos_t - pos_a) // quantize) * quantize
                h = hash((tok_a, tok_t, delta)) & 0xFFFFFFFFFFFFFFFF
            else:
                h = hash((tok_a, tok_t)) & 0xFFFFFFFFFFFFFFFF
            fp.add(h)
    return fp

def jaccard_fp(fp1: set, fp2: set) -> float:
    """Jaccard similarity between two fingerprints."""
    if not fp1 and not fp2:
        return 1.0
    u = len(fp1 | fp2)
    return len(fp1 & fp2) / u if u > 0 else 0.0

# Compute landmark-pair fingerprints for demo
lp_scores = []
for s1, s2, label in sentence_pairs:
    fp1 = fingerprint_landmark_pair(tokenize(s1), idf, TOP_K, WINDOW, QUANTIZE, include_delta=True)
    fp2 = fingerprint_landmark_pair(tokenize(s2), idf, TOP_K, WINDOW, QUANTIZE, include_delta=True)
    score = jaccard_fp(fp1, fp2)
    lp_scores.append(score)

print(f"Landmark-pair scores (with delta): {lp_scores}")

## MinHash and SimHash Baselines

Compute MinHash Jaccard, MinHash Containment, and SimHash similarities.

In [ ]:
def char_shingles(text: str, k: int = 5) -> set:
    """Extract k-character shingles from text."""
    t = text.lower().replace(" ", "")
    return {t[i:i+k] for i in range(max(1, len(t) - k + 1))}

def minhash_jaccard(sh1: set, sh2: set, num_perm: int = 64) -> float:
    """Approximate MinHash Jaccard similarity (fallback: exact)."""
    try:
        from datasketch import MinHash
        m1, m2 = MinHash(num_perm=num_perm), MinHash(num_perm=num_perm)
        for s in sh1:
            m1.update(s.encode())
        for s in sh2:
            m2.update(s.encode())
        return m1.jaccard(m2)
    except ImportError:
        # Exact fallback when datasketch not available
        u = len(sh1 | sh2)
        return len(sh1 & sh2) / u if u > 0 else 0.0

def minhash_containment(sh_query: set, sh_doc: set) -> float:
    """MinHash Containment: |Q ∩ D| / |Q|."""
    if not sh_query:
        return 1.0
    return len(sh_query & sh_doc) / len(sh_query)

def simhash(tokens: list, bits: int = 64) -> int:
    """SimHash fingerprint using bit vectors."""
    v = [0] * bits
    for tok in tokens:
        h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
        for i in range(bits):
            if h & (1 << i):
                v[i] += 1
            else:
                v[i] -= 1
    return sum(1 << i for i in range(bits) if v[i] > 0)

def simhash_similarity(h1: int, h2: int, bits: int = 64) -> float:
    """SimHash similarity via Hamming distance."""
    xor = h1 ^ h2
    hamming = bin(xor).count('1')
    return 1.0 - hamming / bits

# Compute baseline scores
mh_j_scores = []
mh_c_scores = []
sim_scores = []
labels = []

for s1, s2, label in sentence_pairs:
    sh1 = char_shingles(s1, SHINGLE_SIZE)
    sh2 = char_shingles(s2, SHINGLE_SIZE)
    mh_j_scores.append(minhash_jaccard(sh1, sh2, MINHASH_NUM_PERM))
    mh_c_scores.append(minhash_containment(sh1, sh2))
    
    h1 = simhash(tokenize(s1))
    h2 = simhash(tokenize(s2))
    sim_scores.append(simhash_similarity(h1, h2))
    
    labels.append(label)

print(f"MinHash Jaccard: {mh_j_scores}")
print(f"MinHash Containment: {mh_c_scores}")
print(f"SimHash: {sim_scores}")
print(f"Labels: {labels}")

## Evaluation Metrics

Compute recall @ precision ≥ 0.90, F1-optimal, and average precision.

In [ ]:
def recall_at_precision(y_true, scores, min_precision: float = 0.90) -> float:
    """Recall at precision >= min_precision on PR curve."""
    if sum(y_true) == 0:
        return 0.0
    prec, rec, _ = precision_recall_curve(y_true, scores)
    valid = [(p, r) for p, r in zip(prec, rec) if p >= min_precision]
    if not valid:
        return 0.0
    return max(r for _, r in valid)

def f1_optimal(y_true, scores) -> tuple:
    """Best F1 and its threshold."""
    if sum(y_true) == 0:
        return 0.0, 0.5
    prec, rec, thresholds = precision_recall_curve(y_true, scores)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1 = np.where((prec + rec) > 0, 2 * prec * rec / (prec + rec), 0)
    best_idx = np.argmax(f1)
    best_thresh = thresholds[min(best_idx, len(thresholds)-1)]
    return float(f1[best_idx]), float(best_thresh)

# Convert to numpy arrays
y_true = np.array(labels)
scores_lp = np.array(lp_scores)
scores_mh_j = np.array(mh_j_scores)
scores_mh_c = np.array(mh_c_scores)
scores_sim = np.array(sim_scores)

# Compute metrics for each method
methods = {
    'Landmark-Pair': scores_lp,
    'MinHash Jaccard': scores_mh_j,
    'MinHash Containment': scores_mh_c,
    'SimHash': scores_sim,
}

results = {}
for name, scores in methods.items():
    r90 = recall_at_precision(y_true, scores, 0.90)
    f1, thresh = f1_optimal(y_true, scores)
    ap = average_precision_score(y_true, scores) if sum(y_true) > 0 else 0.0
    results[name] = {
        'recall@p90': r90,
        'f1_optimal': f1,
        'threshold': thresh,
        'avg_precision': ap
    }

print("\nMetrics computed successfully")

## Results Summary

Display key metrics and visualization.

In [ ]:
import pandas as pd

# Create results table
results_df = pd.DataFrame(results).T
results_df = results_df.round(4)

print("\n" + "="*70)
print("LANDMARK-PAIR FINGERPRINTING DEMO RESULTS")
print("="*70)
print(f"\nEvaluated on {N_DEMO_EXAMPLES} examples (3 MRPC originals + 2 synthetic edits)")
print(f"Labels distribution: {sum(y_true)} positive, {len(y_true)-sum(y_true)} negative\n")

print(results_df.to_string())

# Per-example predictions
print("\n" + "="*70)
print("PER-EXAMPLE PREDICTIONS")
print("="*70)

pred_data = []
for i, (s1, s2, label) in enumerate(sentence_pairs):
    pred_data.append({
        'Example': i+1,
        'Sentence1': s1[:40] + '...' if len(s1) > 40 else s1,
        'Sentence2': s2[:40] + '...' if len(s2) > 40 else s2,
        'Label': label,
        'LP_Score': lp_scores[i],
        'MH_J_Score': mh_j_scores[i],
        'SimHash_Score': sim_scores[i],
    })

pred_df = pd.DataFrame(pred_data)
print(pred_df.to_string(index=False))

# Load full benchmark results from metadata
print("\n" + "="*70)
print("FULL BENCHMARK RESULTS (from 6076 pairs)")
print("="*70)

benchmark_metrics = {
    'Method': ['Landmark-Pair', 'LP (no delta)', 'MinHash Jaccard', 'MinHash Containment', 'SimHash'],
    'MRPC R@P90': [
        data['metrics_agg']['landmark_pair_mrpc_recall_at_prec90'],
        data['metrics_agg']['landmark_pair_no_delta_mrpc_recall_at_prec90'],
        data['metrics_agg']['minhash_jaccard_mrpc_recall_at_prec90'],
        data['metrics_agg']['minhash_containment_mrpc_recall_at_prec90'],
        data['metrics_agg']['simhash_mrpc_recall_at_prec90'],
    ],
    'Synthetic R@P90': [
        data['metrics_agg']['landmark_pair_synth_recall_at_prec90'],
        data['metrics_agg']['landmark_pair_no_delta_synth_recall_at_prec90'],
        data['metrics_agg']['minhash_jaccard_synth_recall_at_prec90'],
        data['metrics_agg']['minhash_containment_synth_recall_at_prec90'],
        data['metrics_agg']['simhash_synth_recall_at_prec90'],
    ],
    'Overall R@P90': [
        data['metrics_agg']['landmark_pair_all_recall_at_prec90'],
        data['metrics_agg']['landmark_pair_no_delta_all_recall_at_prec90'],
        data['metrics_agg']['minhash_jaccard_all_recall_at_prec90'],
        data['metrics_agg']['minhash_containment_all_recall_at_prec90'],
        data['metrics_agg']['simhash_all_recall_at_prec90'],
    ]
}

bench_df = pd.DataFrame(benchmark_metrics).round(4)
print(bench_df.to_string(index=False))

# Scalability metrics
print("\n" + "="*70)
print("SCALABILITY METRICS")
print("="*70)
scalability = data['metadata']['scalability']
print(f"Avg hashes per passage: {scalability['landmark_pair_avg_hashes_per_passage']:.1f}")
print(f"MinHash hashes per passage: {scalability['minhash_hashes_per_passage']:.0f}")
print(f"Memory @ 1M passages:")
print(f"  Landmark-Pair: {scalability['landmark_pair_memory_1M_MB']:.1f} MB")
print(f"  MinHash: {scalability['minhash_memory_1M_MB']:.1f} MB")
print(f"Retrieval latency (mean): {scalability['retrieval_latency_mean_ms']:.3f} ms")
print(f"Throughput: {scalability['throughput_qps']:.0f} QPS")

print("\n" + "="*70)
print("CONCLUSION")
print("="*70)
print(data['metadata']['novelty_verdict'])

## Visualization: Recall @ Precision ≥ 0.90

In [ ]:
# Plot recall@P90 across methods on full benchmark
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Recall@P90 by dataset type
methods_names = ['LP', 'LP-nodelta', 'MH-J', 'MH-C', 'SimHash']
mrpc_recalls = bench_df['MRPC R@P90'].values
synth_recalls = bench_df['Synthetic R@P90'].values

x = np.arange(len(methods_names))
width = 0.35

ax1.bar(x - width/2, mrpc_recalls, width, label='MRPC (paraphrase)', alpha=0.8)
ax1.bar(x + width/2, synth_recalls, width, label='Synthetic (structural edits)', alpha=0.8)
ax1.set_ylabel('Recall @ Precision ≥ 0.90')
ax1.set_title('Method Comparison: Recall@P90 by Dataset')
ax1.set_xticks(x)
ax1.set_xticklabels(methods_names, rotation=45)
ax1.legend()
ax1.set_ylim([0, 1.05])
ax1.grid(axis='y', alpha=0.3)

# Subplot 2: Overall recall@P90
overall_recalls = bench_df['Overall R@P90'].values
colors = ['#1f77b4' if 'LP' == m.split()[0] else '#ff7f0e' for m in methods_names]
ax2.barh(methods_names, overall_recalls, color=colors, alpha=0.8)
ax2.set_xlabel('Recall @ Precision ≥ 0.90')
ax2.set_title('Overall Recall@P90 (All 6076 Pairs)')
ax2.set_xlim([0, 1.05])
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(overall_recalls):
    ax2.text(v + 0.02, i, f'{v:.3f}', va='center')

plt.tight_layout()
plt.savefig('landmark_pair_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved as landmark_pair_benchmark.png")